In [ ]:
python -m venv rag_env
rag_env\Scripts\activate


pip install pymupdf
pip install pdfplumber
pip install pillow
pip install paddleocr
pip install sentence-transformers
pip install chromadb
pip install langchain
pip install tqdm
pip install unstructured
pip install "unstructured[pdf]"





In [1]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')


c:\Users\neeraj.maurya01\Documents\GE_Ai\QnA_Bot\rag_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:997)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
No sentence-transformers model found with name sentence-transformers/all-MiniLM-L6-v2. Creating a new one with mean pooling.
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:997)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

pipeline.py

In [ ]:
import os
import fitz
import pdfplumber
from tqdm import tqdm
from paddleocr import PaddleOCR
from sentence_transformers import SentenceTransformer
import chromadb
from langchain.text_splitter import RecursiveCharacterTextSplitter

PDF_FOLDER = "data/pdfs"
IMAGE_FOLDER = "output/images"

os.makedirs(IMAGE_FOLDER, exist_ok=True)

# Initialize OCR
ocr = PaddleOCR(use_angle_cls=True, lang='en')

# Embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Vector DB
client = chromadb.Client()
collection = client.create_collection("aveva_docs")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)


def extract_text_pdf(pdf_path):

    text_content = []

    with pdfplumber.open(pdf_path) as pdf:

        for page_num, page in enumerate(pdf.pages):

            text = page.extract_text()

            if text:
                text_content.append(text)

            tables = page.extract_tables()

            for table in tables:
                table_text = "\n".join([" | ".join(row) for row in table if row])
                text_content.append(table_text)

    return "\n".join(text_content)


def ocr_scanned_pages(pdf_path):

    doc = fitz.open(pdf_path)

    ocr_text = []

    for page_index in range(len(doc)):

        page = doc.load_page(page_index)

        pix = page.get_pixmap()

        img_path = f"{IMAGE_FOLDER}/page_{page_index}.png"

        pix.save(img_path)

        result = ocr.ocr(img_path)

        if result:

            page_text = " ".join([line[1][0] for line in result])

            ocr_text.append(page_text)

    return "\n".join(ocr_text)


def chunk_text(text):

    chunks = splitter.split_text(text)

    return chunks


def store_embeddings(chunks, source):

    embeddings = embedding_model.encode(chunks)

    for i, chunk in enumerate(chunks):

        collection.add(
            documents=[chunk],
            embeddings=[embeddings[i]],
            metadatas=[{"source": source}],
            ids=[f"{source}_{i}"]
        )


def process_pdf(pdf_path):

    print(f"Processing: {pdf_path}")

    text = extract_text_pdf(pdf_path)

    if len(text) < 100:
        print("Running OCR...")
        text = ocr_scanned_pages(pdf_path)

    chunks = chunk_text(text)

    store_embeddings(chunks, os.path.basename(pdf_path))


def run_pipeline():

    for file in tqdm(os.listdir(PDF_FOLDER)):

        if file.endswith(".pdf"):

            pdf_path = os.path.join(PDF_FOLDER, file)

            process_pdf(pdf_path)


if __name__ == "__main__":

    run_pipeline()

    print("Pipeline complete")